# 🧠 TALOS — Reindex de Embeddings con Qwen3-Embedding-4B

**Modelo:** `Qwen/Qwen3-Embedding-4B` (4.000 Millones de Parámetros)
**Dimensiones:** `2560` (Precisión Nativa Float16)
**Optimizaciones:** Auto-recuperación ante desconexiones (Checkpoints) y escritura en streaming a Google Drive.

In [ ]:
# 1. Montar Google Drive
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive montado correctamente")

In [ ]:
# 2. Instalación limpia: desinstalamos torchvision (evita conflicto de C++ NMS) e instalamos sentence-transformers
!pip uninstall -y -q torchvision
!pip install -q -U sentence-transformers accelerate


In [ ]:
# 3. Verificación de Acelerador GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ GPU no detectada. En el menú superior ve a: Entorno de ejecución > Cambiar tipo de entorno de ejecución > Selecciona GPU T4")

print(f"✅ GPU activa: {torch.cuda.get_device_name(0)}")
DEVICE = "cuda"

In [ ]:
# 4. Cargar modelo Qwen/Qwen3-Embedding-4B en Float16
from sentence_transformers import SentenceTransformer

MODEL_NAME = "Qwen/Qwen3-Embedding-4B"
DIMENSIONS = 2560
BATCH_SIZE = 128  # Tamaño óptimo para GPU T4 de 15GB

print(f"Cargando {MODEL_NAME} en Float16 nativo...")
model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    trust_remote_code=True,
    model_kwargs={"torch_dtype": torch.float16}
)

def embed_passages(texts):
    with torch.no_grad():
        embeddings = model.encode(
            texts,
            batch_size=BATCH_SIZE,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
    return embeddings.tolist() if hasattr(embeddings, "tolist") else [list(e) for e in embeddings]

# Smoke test
test_vec = embed_passages(["Prueba de verificación semántica TALOS"])[0]
assert len(test_vec) == DIMENSIONS, f"Error: {len(test_vec)} != {DIMENSIONS}"
print(f"✅ Smoke test OK: {DIMENSIONS} dimensiones generadas correctamente en FP16.")

In [ ]:
# 5. Ejecutar vectorización del corpus con auto-reanudación (Checkpoints)
import os, glob, json, time, shutil
from tqdm.notebook import tqdm

DRIVE_FOLDER = "/content/drive/MyDrive/TALOS"
assert os.path.exists(DRIVE_FOLDER), f"No se encontró la carpeta {DRIVE_FOLDER} en tu Google Drive."

# Buscar archivo pending más reciente
pending_files = sorted(glob.glob(f"{DRIVE_FOLDER}/pending_*.jsonl"))
if not pending_files:
    raise FileNotFoundError(f"No se encontró ningún archivo pending_*.jsonl en {DRIVE_FOLDER}")

latest_pending = pending_files[-1]
print(f"📄 Leyendo archivo de pendientes: {latest_pending}")

records = []
with open(latest_pending, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        obj = json.loads(line)
        if "_meta" in obj: continue
        records.append(obj)

print(f"📊 Total de chunks a vectorizar: {len(records):,}")

# Archivo de progreso persistente en Drive
checkpoint_file = f"{DRIVE_FOLDER}/vectors_checkpoint_qwen3_4b.jsonl"
already_done = set()
if os.path.exists(checkpoint_file):
    print(f"⚡ Detectado checkpoint previo. Analizando registros completados...")
    with open(checkpoint_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            o = json.loads(line)
            if "_meta" in o: continue
            already_done.add((o["chunk_id"], o["notebook_id"]))
    print(f"✅ Se reanudará desde el chunk {len(already_done):,}. Chunks ya procesados: {len(already_done):,}")

pending_to_process = [r for r in records if (r["chunk_id"], r["notebook_id"]) not in already_done]
print(f"⏳ Pendientes en esta ejecución: {len(pending_to_process):,}")

# Escribir encabezado de contrato si el archivo es nuevo
if len(already_done) == 0:
    with open(checkpoint_file, "w", encoding="utf-8") as out_fh:
        meta = {
            "_meta": {
                "schema": "talos-vectors-v1",
                "contract": {
                    "schema": "talos-embedding-contract-v1",
                    "model": MODEL_NAME,
                    "model_id": "Qwen3-Embedding-4B",
                    "provider_id": "sentence_transformers",
                    "dimensions": DIMENSIONS,
                    "max_length": 512,
                    "normalize_l2": True,
                    "pooling": "mean",
                    "prefixes": {"query": "", "passage": ""}
                }
            }
        }
        out_fh.write(json.dumps(meta) + "\n")

# Procesamiento por lotes con guardado continuo
with open(checkpoint_file, "a", encoding="utf-8") as out_fh:
    for i in tqdm(range(0, len(pending_to_process), BATCH_SIZE), desc="Vectorizando corpus Qwen3-4B"):
        batch = pending_to_process[i:i+BATCH_SIZE]
        texts = [r.get("text", "") for r in batch]
        vecs = embed_passages(texts)
        for r, vec in zip(batch, vecs):
            out_fh.write(json.dumps({
                "chunk_id": r["chunk_id"],
                "notebook_id": r["notebook_id"],
                "vector": vec
            }) + "\n")
        out_fh.flush()

# Al finalizar completamente, generar el archivo vectors_*.jsonl definitivo
final_vectors_file = f"{DRIVE_FOLDER}/vectors_{time.strftime('%Y%m%dT%H%M%SZ')}.jsonl"
shutil.copy2(checkpoint_file, final_vectors_file)
print(f"\n🎉 ¡VECTORIZACIÓN COMPLETA!")
print(f"Archivo final listo para TALOS: {final_vectors_file}")